# 🧩 Mini-Lab: Walk One Failure Through the Debug Tree

**Module: Debugging Lab 1 — Read the Failure** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Apply** a four-step debugging decision tree: reproduce → localise → shrink → change one thing
2. **Identify** which step of the tree you are on while debugging a real failure
3. **Recognize** that changing more than one thing at a time makes diagnosis impossible

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Debugging Decision Procedure | A repeatable debugging procedure: reproduce the failure, localise it to one pipeline box, shrink the input, then change one thing at a time until the cause is clear. |


## The Debugging Decision Tree

Every time you hit a failure, walk the same four steps **in order**:

```
1. REPRODUCE   →  Can you make it fail again, on demand?
2. LOCALISE    →  Which exact line raises the error?
3. SHRINK      →  What is the smallest input that still fails?
4. CHANGE ONE  →  Fix exactly one thing, then re-run.
       THING       If it still fails, revert and try another.
```

**Why this order?** If you can't reproduce it, you can't prove your fix works. If you haven't localised it, you're guessing. If you haven't shrunk the input, the noise hides the signal. If you change two things at once, you'll never know which one mattered.

Let's walk a concrete failure through all four steps.

## The Buggy Code

Below is a small function that is supposed to compute the **average price** from a list of product records. It has a bug. Our job is to walk the decision tree — not to just eyeball the fix.

In [3]:
def average_price(products):
    """Return the average price from a list of product dicts."""
    total = 0
    for p in products:
        total += p["price"]
    return total / len(products)


# Sample data — looks reasonable
catalog = [
    {"name": "Widget",  "price": 9.99},
    {"name": "Gadget",  "price": 24.50},
    {"name": "Gizmo",   "price": "19.99"},   # <-- came from a CSV import
    {"name": "Doohick", "price": 4.99},
]

## Step 1 — REPRODUCE

> 🧩 **The debugging decision tree.** The four-step tree from the video; here we walk step one.

> *"Can I make it fail again, right now, on demand?"*

Run the function and confirm the error appears.

In [5]:
try:
    result = average_price(catalog)
    print(f"Average price: {result}")
except TypeError as e:
    print(f"✅ Reproduced! Error: {e}")

✅ Reproduced! Error: unsupported operand type(s) for +=: 'float' and 'str'


Good — the error reproduces every time. We have a `TypeError`. That means we can trust that once we fix it, a clean run proves the fix.

**Decision tree checkpoint:** ✅ Reproduce — done. Move to **Localise**.

## Step 2 — LOCALISE

> *"Which exact line raises the error?"*

We read the traceback. Python tells us which line failed. Let's get the full traceback to localise it.

In [ ]:
import traceback

try:
    average_price(catalog)
except TypeError:
    traceback.print_exc()

The traceback points to the line `total += p["price"]`. The error is something like:

```
TypeError: unsupported operand type(s) for +=: 'int' and 'str'
```

So `p["price"]` is a **string** for at least one product. We've localised the line. Now — which item?

**Decision tree checkpoint:** ✅ Localise — done. Move to **Shrink**.

## Step 3 — SHRINK

> *"What is the smallest input that still fails?"*

We have 4 items. Let's find the **minimum** list that still triggers the error. We test with each item alone.

In [ ]:
for i, product in enumerate(catalog):
    try:
        average_price([product])
        print(f"  Item {i} ({product['name']}): OK")
    except TypeError:
        print(f"❌ Item {i} ({product['name']}): FAILS — price is {type(product['price']).__name__} = {product['price']!r}")

The minimal failing input is a **single-element list** containing the "Gizmo" product whose `price` is the string `"19.99"` instead of a float.

**Decision tree checkpoint:** ✅ Shrink — done. Move to **Change One Thing**.

## Step 4 — CHANGE ONE THING

> *"Fix exactly one thing, then re-run. If it still fails, revert and try another."*

We know the root cause: `price` can be a string. We have **one** change to make: convert `price` to `float` inside the loop. We change **only** that line and re-run on the full catalog.

In [ ]:
def average_price_fixed(products):
    """Return the average price — handles string prices."""
    total = 0
    for p in products:
        total += float(p["price"])   # ← the ONE change
    return total / len(products)


result = average_price_fixed(catalog)
print(f"Average price: {result}")

The function now returns the correct average. We changed **one thing** (`int` addition → `float()` conversion), re-ran, and the failure is gone.

**Decision tree checkpoint:** ✅ Change One Thing — done. Bug fixed.

## The Full Walk — Recap

Let's print the decision tree with our answers filled in, so you see the whole procedure in one place.

In [ ]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

md("""
| Step | Question | Our Answer |
|------|----------|------------|
| **1. Reproduce** | Can I make it fail on demand? | Yes — `average_price(catalog)` raises `TypeError` every time |
| **2. Localise** | Which line fails? | `total += p["price"]` — adding `int` and `str` |
| **3. Shrink** | Smallest failing input? | `[{"name": "Gizmo", "price": "19.99"}]` — one item |
| **4. Change One Thing** | What single change fixes it? | Wrap with `float()`: `total += float(p["price"])` |
""")

## 🎯 Summary

### Key Takeaways

1. **Reproduce first** — if you can't trigger the bug on demand, you can't prove your fix works
2. **Localise before guessing** — read the traceback to find the exact failing line instead of scanning the whole file
3. **Shrink the input** — strip away everything that doesn't matter until you have the minimal failing case
4. **Change one thing at a time** — if you change two things and the bug disappears, you don't know which change fixed it (and the other change might introduce a new bug)
5. **The tree is repeatable** — follow the same four steps for every failure, regardless of language or domain